In [1]:
import yfinance as yf

# Example: Apple stock
ticker = "AAPL"

data = yf.download(
    tickers=ticker,
    interval="5m",
    period="60d"   # max for 5m data
)

data = data.rename(columns=str.lower)
data = data[['open', 'high', 'low', 'close', 'volume']]

# print(data.head())

[*********************100%***********************]  1 of 1 completed


In [2]:
# print(data.info())
# print(data.describe())

In [3]:
# Make a copy to avoid modifying original
df = data.copy()
df.columns = df.columns.get_level_values(0)

threshold = 0.001  # 0.1% move

# Shift by 10 minutes (2 rows)
df['future_close'] = df['close'].shift(-2)

# Binary classification target
df['future_return'] = (df['future_close'] / df['close']) - 1
df['target'] = (df['future_return'] > threshold).astype(int)

# Drop last rows without future data
df = df.dropna()
print(df['target'].value_counts())

target
0    3432
1    1246
Name: count, dtype: int64


In [4]:
import numpy as np

WINDOW = 10

X = []
y = []

for i in range(len(df) - WINDOW):
    X.append(df[['open','high','low','close','volume']].iloc[i:i+WINDOW].values)
    y.append(df['target'].iloc[i+WINDOW])

X = np.array(X)
y = np.array(y)

print(X.shape)  # (samples, WINDOW, 5)

(4668, 10, 5)


In [5]:
import numpy as np

def transform_window(window):
    """
    window shape: (30, 5)
    columns: [open, high, low, close, volume]
    
    returns transformed window of shape (30, 5)
    """
    window = window.astype(np.float32).copy()
    
    o = window[:, 0]
    h = window[:, 1]
    l = window[:, 2]
    c = window[:, 3]
    v = window[:, 4]
    
    eps = 1e-8
    
    # 1. Close-to-close returns
    close_ret = np.zeros_like(c)
    close_ret[1:] = (c[1:] / (c[:-1] + eps)) - 1.0
    
    # 2. Candle shape relative to close
    open_rel = (o / (c + eps)) - 1.0
    high_rel = (h / (c + eps)) - 1.0
    low_rel  = (l / (c + eps)) - 1.0
    
    # 3. Log volume
    log_vol = np.log1p(v)
    
    # 4. Normalize volume within the window
    vol_mean = log_vol.mean()
    vol_std = log_vol.std() + eps
    vol_z = (log_vol - vol_mean) / vol_std
    
    transformed = np.column_stack([
        close_ret,
        open_rel,
        high_rel,
        low_rel,
        vol_z
    ])
    
    return transformed.astype(np.float32)

In [6]:
def transform_window_v2(window):
    """
    window shape: (WINDOW, 5)
    columns: [open, high, low, close, volume]
    
    returns shape: (WINDOW, 8)
    """
    window = window.astype(np.float32).copy()
    
    o = window[:, 0]
    h = window[:, 1]
    l = window[:, 2]
    c = window[:, 3]
    v = window[:, 4]
    
    eps = 1e-8
    
    close_ret = np.zeros_like(c)
    close_ret[1:] = (c[1:] / (c[:-1] + eps)) - 1.0
    
    open_rel = (o / (c + eps)) - 1.0
    high_rel = (h / (c + eps)) - 1.0
    low_rel  = (l / (c + eps)) - 1.0
    
    body = (c - o) / (o + eps)
    upper_wick = (h - np.maximum(o, c)) / (c + eps)
    lower_wick = (np.minimum(o, c) - l) / (c + eps)
    
    log_vol = np.log1p(v)
    vol_z = (log_vol - log_vol.mean()) / (log_vol.std() + eps)
    
    transformed = np.column_stack([
        close_ret,
        open_rel,
        high_rel,
        low_rel,
        body,
        upper_wick,
        lower_wick,
        vol_z
    ])
    
    return transformed.astype(np.float32)

In [7]:
X_transformed = np.array([transform_window_v2(window) for window in X], dtype=np.float32)
print(X_transformed.shape)   # (2690, WINDOW, 8)

(4668, 10, 8)


In [10]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# 1. Assume you already have:
#    X -> shape (2690, WINDOW, 8)
#    y -> shape (2690,)
# -----------------------------

# Example:
# X = np.array(X, dtype=np.float32)
# y = np.array(y, dtype=np.float32)

X = X_transformed.astype(np.float32)
y = y.astype(np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

# -----------------------------
# 2. Train/test split
# -----------------------------
# For time series, a chronological split is usually better than random.
# This keeps the earlier data for training and later data for testing.

n = len(X)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

# -----------------------------
# 3. Dataset class
# -----------------------------
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].transpose(0, 1)   # (features, time)
        y = self.y[idx]
        return x, y

train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# -----------------------------
# 4. CNN model
# -----------------------------
class StockCNN(nn.Module):
    def __init__(self, in_channels=8):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.2),

            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),

            nn.MaxPool1d(2),   # 30 -> 15
            nn.Dropout(0.3),

            nn.Conv1d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),

            nn.AdaptiveAvgPool1d(1)   # output shape: (batch, 32, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StockCNN().to(device)

# print(model)


# -----------------------------
# 5. Loss and optimizer
# -----------------------------
# BCEWithLogitsLoss expects raw logits, not sigmoid probabilities.

num_pos = (y_train == 1).sum()
num_neg = (y_train == 0).sum()

pos_weight = torch.tensor([num_neg / num_pos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# -----------------------------
# 6. Evaluation function
# -----------------------------
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    
    total_loss = 0.0
    all_probs = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    
    return avg_loss, acc, np.array(all_probs), np.array(all_preds), np.array(all_labels)

# -----------------------------
# 7. Training loop
# -----------------------------
best_val_loss = float("inf")
patience = 7
patience_counter = 0
best_model_state = None
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_loss, val_acc, _, _, _ = evaluate_model(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # if val_loss < best_val_loss:
    #     best_val_loss = val_loss
    #     patience_counter = 0
    #     best_model_state = model.state_dict()
    # else:
    #     patience_counter += 1
    #     if patience_counter >= patience:
    #         print("Early stopping triggered.")
    #         break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

# -----------------------------
# 8. Final evaluation
# -----------------------------
test_loss, test_acc, probs, preds, labels = evaluate_model(model, test_loader, criterion, device)

print("\nFinal Test Loss:", test_loss)
print("Final Test Accuracy:", test_acc)
print("\nConfusion Matrix:")
print(confusion_matrix(labels, preds))
print("\nClassification Report:")
print(classification_report(labels, preds, digits=4))

#saving model for server to access
torch.save(model.state_dict(), "saved_models/stock_cnn.pt")

X shape: (4668, 10, 8)
y shape: (4668,)
Train shape: (3267, 10, 8) (3267,)
Test shape: (701, 10, 8) (701,)
Epoch 1/30 | Train Loss: 1.0170 | Val Loss: 0.9808 | Val Acc: 0.7471
Epoch 2/30 | Train Loss: 1.0140 | Val Loss: 0.9799 | Val Acc: 0.7357
Epoch 3/30 | Train Loss: 1.0096 | Val Loss: 0.9783 | Val Acc: 0.6829
Epoch 4/30 | Train Loss: 1.0089 | Val Loss: 0.9782 | Val Acc: 0.6000
Epoch 5/30 | Train Loss: 1.0097 | Val Loss: 0.9776 | Val Acc: 0.5771
Epoch 6/30 | Train Loss: 1.0063 | Val Loss: 0.9777 | Val Acc: 0.5586
Epoch 7/30 | Train Loss: 1.0052 | Val Loss: 0.9774 | Val Acc: 0.5471
Epoch 8/30 | Train Loss: 1.0046 | Val Loss: 0.9776 | Val Acc: 0.5300
Epoch 9/30 | Train Loss: 1.0076 | Val Loss: 0.9791 | Val Acc: 0.5300
Epoch 10/30 | Train Loss: 1.0078 | Val Loss: 0.9785 | Val Acc: 0.5314
Epoch 11/30 | Train Loss: 1.0044 | Val Loss: 0.9785 | Val Acc: 0.5229
Epoch 12/30 | Train Loss: 1.0020 | Val Loss: 0.9780 | Val Acc: 0.5286
Epoch 13/30 | Train Loss: 1.0015 | Val Loss: 0.9787 | Val Acc: